In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
spark = SparkSession.builder.appName('15').getOrCreate()

In [3]:
data = spark.read.csv('C:/Users/User/Downloads/realestate.csv',
                      header=True,inferSchema=True)

In [4]:
data.show(3)

+---+---------------+--------+-------------+-----------------------+--------+---------+---------------+
| No|TransactionDate|HouseAge|DistanceToMRT|NumberConvenienceStores|Latitude|Longitude|PriceOfUnitArea|
+---+---------------+--------+-------------+-----------------------+--------+---------+---------------+
|  1|       2012.917|    32.0|     84.87882|                     10|24.98298|121.54024|           37.9|
|  2|       2012.917|    19.5|     306.5947|                      9|24.98034|121.53951|           42.2|
|  3|       2013.583|    13.3|     561.9845|                      5|24.98746|121.54391|           47.3|
+---+---------------+--------+-------------+-----------------------+--------+---------+---------------+
only showing top 3 rows



In [5]:
data.printSchema()

root
 |-- No: integer (nullable = true)
 |-- TransactionDate: double (nullable = true)
 |-- HouseAge: double (nullable = true)
 |-- DistanceToMRT: double (nullable = true)
 |-- NumberConvenienceStores: integer (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- PriceOfUnitArea: double (nullable = true)



- **HouseAge, DistanceToMRT, NumberConvenienceStores**을 특징 `X`(feature)로 하고, **PriceOfUnitArea**를 예측값 `y`로 하는 **Random Forest Regressor** 생성

## 1. `VectorAssemble`
- **HouseAge, DistanceToMRT, NumberConvenienceStores** 3개의 column을 **하나의 feature vector column**으로 변환해야 함
- `pyspark`의 `ml`에서는 `X`가 복수의 column이 아닌 **하나의 column에 list 형태로 feature vector**가 저장된 형태여야 함
- `VectorAssemble`은 이러한 변환에 사용됨

- `VectorAssembler()`
> - `setInputCols()`: **필수**, 특징(`X`)로 사용할 column의 이름을 list로 지정 (또는 parameter `inputCols`)
> - `setOutputCol()`: vector로 결합된 후의 column 이름을 지정 (또는 parameter `outputCol`)

In [6]:
va = VectorAssembler().\
setInputCols(['HouseAge', 'DistanceToMRT', 'NumberConvenienceStores']).\
setOutputCol('features')

- `VectorAssembler().transform(DataFrame)`: 입력된 `DataFrame`을 지정한 형태로 변환함
- `ml`을 사용할 때 필요한 column은 `X`를 의미하는 **vector** (위의 경우, `features`)와 `y` (위의 경우, `PriceOfUnitArea`)만 필요함

In [7]:
pyml_data = va.transform(data).select('features','PriceOfUnitArea')
pyml_data.show(3)
pyml_data.printSchema()

+--------------------+---------------+
|            features|PriceOfUnitArea|
+--------------------+---------------+
|[32.0,84.87882,10.0]|           37.9|
| [19.5,306.5947,9.0]|           42.2|
| [13.3,561.9845,5.0]|           47.3|
+--------------------+---------------+
only showing top 3 rows

root
 |-- features: vector (nullable = true)
 |-- PriceOfUnitArea: double (nullable = true)



## 2. `train/test` data 분리
- `DataFrame.radomSplit([a,b])`: `a:b`의 비율로 **DataFrame**을 2개로 분리하여 **list type**으로 반환

In [8]:
split_data = pyml_data.randomSplit([0.7,0.3],seed=1)

In [9]:
print(type(split_data))
print(len(split_data))
print(type(split_data[0]))

<class 'list'>
2
<class 'pyspark.sql.dataframe.DataFrame'>


In [10]:
train_data = split_data[0]
test_data = split_data[1]

In [11]:
# split_data의 각 요소는 pyml_data와 schema가 동일함
train_data.show(5)
train_data.printSchema()

+------------------+---------------+
|          features|PriceOfUnitArea|
+------------------+---------------+
|[0.0,185.4296,0.0]|           37.9|
|[0.0,185.4296,0.0]|           45.5|
|[0.0,185.4296,0.0]|           52.2|
|[0.0,185.4296,0.0]|           55.2|
|[0.0,208.3905,6.0]|           45.7|
+------------------+---------------+
only showing top 5 rows

root
 |-- features: vector (nullable = true)
 |-- PriceOfUnitArea: double (nullable = true)



## 3. `RandomForestRegressor`
- `featuresCol, labelCol`: `X`와 `y`는 필수로 지정
- `predictionCol`: 예측된 결과값이 저장될 column명 (디폴트: `prediction`)
- `numTrees`: decision tree의 수 (디폴트: 20)
- `minInstancesPerNode`: leaf node로 간주하는 최소 data의 수(값이 크면 단순해지고, overfitting을 방지할 수 있으나, 너무 크면 underfitting의 위험이 있음, 디폴트: 1)
- `maxDepth`: decision tree의 최대층 (디폴트: 5)
- `bootstrap`: bootstrap 실행 여부 (디폴트: True)
- `featureSubsetStrategy`: 랜덤하게 선정할 특징의 수를 결정하는 방식 (디폴트: `auto`(regressor의 경우 특징 수의 1/3 선택, classifier는 특징 수의 sqrt를 적용한 결과))
- 기타 등등, `sklearn`의 `RandomForestRegressor`와 유사

In [12]:
rf = RandomForestRegressor(featuresCol='features',
                           labelCol='PriceOfUnitArea',
                           predictionCol='predictedPrice',
                           numTrees=200,
                           minInstancesPerNode=3,
                           featureSubsetStrategy='onethird')

- `RandomForestRegressor.fit(data)`: data는 `featureCol`과 `labelCol`을 모두 포함한 `DataFrame`, 학습된 `RandomForestRegrssionModel`을 반환

In [13]:
rf_model = rf.fit(train_data)
# rf와 rf_model은 다른 클래스임

In [14]:
print(type(rf), type(rf_model))

<class 'pyspark.ml.regression.RandomForestRegressor'> <class 'pyspark.ml.regression.RandomForestRegressionModel'>


## 4. `transform` (predict, 예측결과 확인하기)
- `RandomForestRegressionModel.transform(dataset)`
> - 입력(`dataset`: `featuresCol, labelCol`을 포함)에 대한 예측값 계산
> - **반환**: `featuresCol, labelCol, predictionCol`으로 구성된 **DataFrame**

In [15]:
train_data.show(2)
test_data.show(2)
# featuresCol='features', labelCol='PriceOfUnitArea'

+------------------+---------------+
|          features|PriceOfUnitArea|
+------------------+---------------+
|[0.0,185.4296,0.0]|           37.9|
|[0.0,185.4296,0.0]|           45.5|
+------------------+---------------+
only showing top 2 rows

+------------------+---------------+
|          features|PriceOfUnitArea|
+------------------+---------------+
|[0.0,185.4296,0.0]|           55.3|
|[0.0,208.3905,6.0]|           44.0|
+------------------+---------------+
only showing top 2 rows



In [16]:
train_pred = rf_model.transform(train_data)
test_pred = rf_model.transform(test_data)

In [17]:
train_pred.show(2)
test_pred.show(2)

+------------------+---------------+-----------------+
|          features|PriceOfUnitArea|   predictedPrice|
+------------------+---------------+-----------------+
|[0.0,185.4296,0.0]|           37.9|47.63753283138472|
|[0.0,185.4296,0.0]|           45.5|47.63753283138472|
+------------------+---------------+-----------------+
only showing top 2 rows

+------------------+---------------+------------------+
|          features|PriceOfUnitArea|    predictedPrice|
+------------------+---------------+------------------+
|[0.0,185.4296,0.0]|           55.3| 47.63753283138472|
|[0.0,208.3905,6.0]|           44.0|53.624567575412954|
+------------------+---------------+------------------+
only showing top 2 rows



## 5. `RegressionEvaluator` 성능평가
- 예측결과 column(`predictionCol`)과 실제 값 column(`labelCol`), **metric**(`mse, mae, r2`)을 지정하여 regression의 성능을 평가함

In [18]:
rf_eval = RegressionEvaluator(predictionCol='predictedPrice',
                              labelCol='PriceOfUnitArea',
                              metricName='r2')

- `RegressionEvaluator.evaluate(dataset)`: `predictionCol`과 `labelCol`을 모두 포함한 `DataFrame`을 인자로 전달

In [19]:
print('Train R2: ', rf_eval.evaluate(train_pred))
print('Test R2 : ', rf_eval.evaluate(test_pred))

Train R2:  0.7151587312686905
Test R2 :  0.7602325755857289
